In [ ]:
# %%bash
# python -m pip install --upgrade pip
# pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# pip install "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes transformers datasets torch torchvision

In [1]:
import torch
from trl import SFTTrainer
from datasets import Dataset
from transformers import TrainingArguments
from unsloth import FastLanguageModel, is_bfloat16_supported

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


In [2]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-2-9b-it-bnb-4bit",
    max_seq_length=8192,
    dtype=None,
    load_in_4bit=True,
)

==((====))==  Unsloth: Fast Gemma2 patching release 2024.7
   \\   /|    GPU: NVIDIA RTX A4500. Max memory: 19.698 GB. Platform = Linux.
O^O/ \_/ \    Pytorch: 2.3.0+cu121. CUDA = 8.6. CUDA Toolkit = 12.1.
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.26.post1. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth


In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

Unsloth 2024.7 patched 42 layers with 42 QKV layers, 42 O layers and 42 MLP layers.


In [4]:
prompt = """You are a professional eCommerce manager. Your task is to create a great product description in specific format based on the provided product data.

Steps to complete the task:
1. Read the product data.
2. Create a great product description based on the product data.
3. Format the product description correctly.
4. Return only the product description and nothing else.

<format>
    <product name/>
    <price/> | <sku/>

    <short product description/>

    Why you'll love it:
    - <important product features/>
    - <important product features/>
    - ...

    <call to action/>
</format>"""

train_prompt = (
    prompt
    + """

<product-data>
{}
</product-data>

New product description:
{}"""
)

In [5]:
def formatting_prompts_func(examples):
    texts = []
    for input, output in zip(examples["features"], examples["text"]):
        text = (
            train_prompt.format(input.strip(), output.strip())
        )
        texts.append(text)
    return {
        "text": texts,
    }

In [6]:
dataset = Dataset.from_json("/root/asos-dataset.json")

In [7]:
print(dataset[0]["features"])

print("\n-------\n")

print(dataset[0]["text"])

- Name: Nike Swoosh graphic slim crop t-shirt in black
- Price: Now £22.50
- SKU: 1154894
- Description: Product Details: T-shirts by Nike. Act casual and cool in this slim-fit crop top featuring the iconic Nike Swoosh graphic. Crafted from soft, breathable fabric for all-day comfort.

-------

Nike Swoosh Graphic Slim Crop T-Shirt in Black
Now £22.50 | SKU: 1154894

Embrace effortless style with the Nike Swoosh Graphic Slim Crop T-Shirt. This sleek top combines a classic design with modern comfort, making it perfect for your everyday look.

Why you'll love it:
- Iconic Style: The bold Nike Swoosh graphic adds a touch of athletic flair to any outfit.
- Flattering Fit: The slim crop silhouette hugs your figure and shows off your shape.
- All-Day Comfort: Crafted from soft, breathable fabric, this tee keeps you feeling fresh and confident all day long.

Whether you're hitting the gym or running errands, this versatile top is a must-have for any wardrobe. 

Get yours now for only £22.50!


In [8]:
dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
)

In [9]:
print(dataset[0]["text"])

You are a professional eCommerce manager. Your task is to create a great product description in specific format based on the provided product data.

Steps to complete the task:
1. Read the product data.
2. Create a great product description based on the product data.
3. Format the product description correctly.
4. Return only the product description and nothing else.

<format>
    <product name/>
    <price/> | <sku/>

    <short product description/>

    Why you'll love it:
    - <important product features/>
    - <important product features/>
    - ...

    <call to action/>
</format>

<product-data>
- Name: Nike Swoosh graphic slim crop t-shirt in black
- Price: Now £22.50
- SKU: 1154894
- Description: Product Details: T-shirts by Nike. Act casual and cool in this slim-fit crop top featuring the iconic Nike Swoosh graphic. Crafted from soft, breathable fabric for all-day comfort.
</product-data>

New product description:
Nike Swoosh Graphic Slim Crop T-Shirt in Black
Now £22.50 | 

In [10]:
dataset.shuffle(seed=42)
dataset.set_format("torch")

In [11]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA RTX A4500. Max memory = 19.698 GB.
6.576 GB of memory reserved.


In [12]:
trainingArgs = TrainingArguments(
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=50,
        optim="adamw_8bit",
        warmup_steps=25,
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="output",
        overwrite_output_dir=True,
    )

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=8192,
    dataset_num_proc=2,
    args=trainingArgs
)

In [14]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 716 | Num Epochs = 3
O^O/ \_/ \    Batch size per device = 8 | Gradient Accumulation steps = 1
\        /    Total batch size = 8 | Total steps = 270
 "-____-"     Number of trainable parameters = 54,018,048


Step,Training Loss
50,0.672300
100,0.344000
150,0.282000
200,0.246200
250,0.203700


In [15]:
# Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

1134.7444 seconds used for training.
18.91 minutes used for training.
Peak reserved memory = 13.217 GB.
Peak reserved memory for training = 6.641 GB.
Peak reserved memory % of max memory = 67.098 %.
Peak reserved memory for training % of max memory = 33.714 %.


In [16]:
FastLanguageModel.for_inference(model)

In [17]:
test_prompt = prompt + """

<product-data>
{}
</product-data>"""

In [18]:
input = """
- Name: NIKE Air Max 90
- Price: $120
- SKU: 1195783
- Description: The Nike Air Max 90 stays true to its OG roots with the iconic Waffle sole, stitched overlays and classic TPU accents. Fresh colors give a modern look while Max Air cushioning adds comfort to your journey.
""".strip()

test_prompt = test_prompt.format(input)
print(test_prompt)

You are a professional eCommerce manager. Your task is to create a great product description in specific format based on the provided product data.

Steps to complete the task:
1. Read the product data.
2. Create a great product description based on the product data.
3. Format the product description correctly.
4. Return only the product description and nothing else.

<format>
    <product name/>
    <price/> | <sku/>

    <short product description/>

    Why you'll love it:
    - <important product features/>
    - <important product features/>
    - ...

    <call to action/>
</format>

<product-data>
- Name: NIKE Air Max 90
- Price: $120
- SKU: 1195783
- Description: The Nike Air Max 90 stays true to its OG roots with the iconic Waffle sole, stitched overlays and classic TPU accents. Fresh colors give a modern look while Max Air cushioning adds comfort to your journey.
</product-data>


In [19]:
inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=1028, use_cache=True)
output = tokenizer.batch_decode(outputs)[0]

In [22]:
print(output.split("New product description:")[1].replace("<end_of_turn><eos>", "").strip())

Nike Air Max 90
$120.00 | SKU: 1195783

Experience the legendary comfort and style of the Nike Air Max 90. This iconic shoe stays true to its roots with a classic Waffle sole, stitched overlays, and TPU accents that define its timeless design.

Why you'll love it:
- Iconic Style: The Air Max 90's bold silhouette and fresh colorways make a statement wherever you go.
- Superior Comfort: Max Air cushioning provides all-day support and a smooth ride.
- Built to Last: Durable construction ensures these shoes will be your go-to choice for years to come.

Relive the legacy of innovation with the Nike Air Max 90. Order yours today for $120.00 and step into comfort and style.


In [21]:
# If you want to save the model in a GGUF format, you can uncomment the following lines

# model.save_pretrained_gguf(
#     "model",
#     tokenizer,
# )